# Galaxy gallery

Two galleries of nearby galaxies, each built from one CasJobs query and SkyServer's image cutout service:

1. **Random galaxies:** a random sample of galaxies that look large on the sky.
2. **Color–magnitude gallery:** the nearest galaxy in each cell of the color–magnitude diagram, which sorts galaxies by color and luminosity, and with them by shape.

Database ideas: a random sample with `ORDER BY NEWID()`, a server-side function inside a query, window functions that pick one row from each group, and an image service driven by values from the catalog.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.patches import Rectangle
from SciServer import CasJobs, SkyServer

import settings

# The queries in this notebook need no SciServer login. Silence the
# warning SciScript gives about the missing login token.
warnings.filterwarnings("ignore", message="In Authentication.getToken")

## 1. Random galaxies

`SELECT TOP 15 ...` returns $15$ rows, but SQL does not say which. Without `ORDER BY`, the server returns the first rows its query plan happens to produce: not a random sample, and not guaranteed to be the same rows twice.

`ORDER BY NEWID()` gives every matching row a fresh random identifier and sorts the rows by it, so `TOP` then takes a random sample. The server has to number and sort every matching row; that is quick for the ten thousand rows here and slow for a whole table.

The galaxies come from the **Main Galaxy Sample** of SDSS-I/II (the `GALAXY` bit of `legacy_target1`), in the nearby redshift range of the parameter set. `p.clean = 1` keeps only galaxies whose photometry the SDSS pipeline marks as clean; it removes about a quarter of the sample. The last condition keeps galaxies large enough on the sky to fill a cutout with the camera's own pixels (explained below), about one galaxy in five.

In [ ]:
random_limits = settings.parameter_set("random_gallery")
min_radius = (
    settings.SDSS_PIXEL_SCALE
    * random_limits["width"]
    / settings.CUTOUT_FIELD_OF_VIEW
)
random_query = f"""
SELECT TOP {random_limits["size"]}
    s.ra, s.dec, s.z, p.petroR90_r
FROM SpecObj AS s
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
WHERE
    (s.legacy_target1 & dbo.fPrimTarget('GALAXY')) > 0
    AND s.class = 'GALAXY'
    AND s.zWarning = 0
    AND p.clean = 1
    AND s.z BETWEEN {random_limits["z_min"]} AND {random_limits["z_max"]}
    AND p.petroR90_r >= {min_radius:.2f}
ORDER BY NEWID()
"""
print(random_query)

In [ ]:
%%time
random_galaxies = CasJobs.executeQuery(
    sql=random_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
random_galaxies

### Image cutouts

`SkyServer.getJpegImgCutout` asks SkyServer for a color JPEG of any patch of the SDSS sky, made from the $g$, $r$ and $i$ images. `scale` sets the pixel size in arcseconds and `width` and `height` the image size in pixels, so the field of view is `scale` times `width`.

The function below frames each galaxy by its size in the catalog: the field of view spans `settings.CUTOUT_FIELD_OF_VIEW` times `petroR90_r`, the radius that holds $90\,\%$ of the galaxy's light. Pixels finer than the camera's $0.396''$ would only interpolate, so `scale` never goes below that. This is where `min_radius` above comes from: a galaxy smaller than `min_radius` would need finer pixels to fill the frame.

In [ ]:
def cutout(galaxy, width):
    """JPEG cutout of the sky around a galaxy, framed by its size."""
    field_of_view = settings.CUTOUT_FIELD_OF_VIEW * galaxy.petroR90_r
    scale = max(settings.SDSS_PIXEL_SCALE, field_of_view / width)
    return SkyServer.getJpegImgCutout(
        ra=galaxy.ra,
        dec=galaxy.dec,
        scale=scale,
        width=width,
        height=width,
        dataRelease=settings.SKYSERVER_RELEASE,
    )

Each cutout is a separate web request, so the time below grows with the number of galaxies.

In [ ]:
%%time
random_images = [
    cutout(galaxy, random_limits["width"])
    for galaxy in random_galaxies.itertuples()
]

In [ ]:
columns = random_limits["columns"]
rows = math.ceil(len(random_galaxies) / columns)

with plt.style.context(settings.DARK_STYLE):
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(
            settings.FIGURE_WIDTH,
            settings.FIGURE_WIDTH * rows / columns,
        ),
        squeeze=False,
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, image, galaxy in zip(
        axes.flat, random_images, random_galaxies.itertuples()
    ):
        ax.imshow(image)
        ax.text(0.04, 0.04, f"$z = {galaxy.z:.3f}$", transform=ax.transAxes)
    plt.show()

Run the query cell again for a different sample.

## 2. Color–magnitude gallery

The **color–magnitude diagram** plots each galaxy's rest-frame color against its absolute magnitude, which measures luminosity on a logarithmic scale (more negative is brighter). Nearby galaxies split into two groups in it: the **red sequence** of mostly luminous red galaxies and the **blue cloud** of bluer ones. The gallery below puts a picture of a real galaxy into each cell of the diagram, to show what the two groups look like.

The absolute magnitude is

$$M_r = r - \mathrm{DM}(z) - K_r,$$

where $r$ is the apparent magnitude, $\mathrm{DM}(z)$ the distance modulus of the galaxy's redshift, and $K_r$ the k-correction stored in `Photoz` (the same k-corrections give the rest-frame color). `Photoz` computes them at the photometric redshift rather than the spectroscopic one; the two typically differ by $0.01$–$0.02$ here, which shifts $g - r$ by about $0.015$, a small fraction of a cell. The database computes the distance modulus itself: SkyServer has a family of cosmology functions, and `dbo.fCosmoDistanceModulus` takes the redshift and the cosmological parameters $\Omega_\mathrm{m}$, $\Omega_\Lambda$, $\Omega_\mathrm{r}$, $w$ and $h$, and returns the distance modulus for that cosmology. The cell below writes that call with the parameters of `settings.COSMOLOGY`, so that the server's distances match the ones astropy computes in the other notebooks.

In [ ]:
gallery_limits = settings.parameter_set("color_magnitude_gallery")
cosmology = settings.COSMOLOGY
radiation_density = cosmology.Ogamma0 + cosmology.Onu0
distance_modulus = (
    f"dbo.fCosmoDistanceModulus(s.z, {cosmology.Om0}, {cosmology.Ode0}, "
    f"{radiation_density}, {cosmology.w(0)}, {cosmology.h})"
)
print(distance_modulus)

### One galaxy per cell

The query is built in three named steps with `WITH`:

1. `galaxies`: the sample, with absolute magnitude and rest-frame color computed on the server.
2. `cells`: the cell of the diagram each galaxy falls in, as a magnitude bin and a color bin number. `FLOOR` rounds down.
3. `ranked`: two **window functions**. `GROUP BY` collapses each group into one row; a window function instead keeps every row and adds a value computed over the row's group, the *partition*. `ROW_NUMBER() OVER (PARTITION BY mag_bin, color_bin ORDER BY z)` numbers the galaxies of each cell from the nearest outwards, and `COUNT(*) OVER (PARTITION BY mag_bin, color_bin)` writes the number of galaxies in the cell into each row.

The final `SELECT` keeps the first galaxy of every cell that holds at least `min_count` galaxies. The server reads tens of thousands of galaxies and sends back one row per cell. Ordering each cell by redshift picks its nearest galaxy, which gives the sharpest picture of each kind and the same gallery on every run; ordering by `NEWID()` would pick a random one instead.

In [ ]:
gallery_query = """
WITH galaxies AS (
    SELECT
        s.specObjID, s.ra, s.dec, s.z, p.petroR90_r,
        p.dered_r - pz.kcorrR - {distance_modulus} AS abs_r,
        (p.dered_g - pz.kcorrG) - (p.dered_r - pz.kcorrR) AS rest_gr
    FROM SpecObj AS s
        JOIN PhotoObj AS p ON p.objID = s.bestObjID
        JOIN Photoz AS pz ON pz.objID = s.bestObjID
    WHERE
        (s.legacy_target1 & dbo.fPrimTarget('GALAXY')) > 0
        AND s.class = 'GALAXY'
        AND s.zWarning = 0
        AND p.clean = 1
        AND s.z BETWEEN {z_min} AND {z_max}
),
cells AS (
    SELECT
        *,
        FLOOR((abs_r - ({mag_min})) / {mag_step}) AS mag_bin,
        FLOOR((rest_gr - {color_min}) / {color_step}) AS color_bin
    FROM galaxies
    WHERE
        abs_r >= {mag_min} AND abs_r < {mag_max}
        AND rest_gr >= {color_min} AND rest_gr < {color_max}
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY mag_bin, color_bin ORDER BY z
        ) AS rank_in_cell,
        COUNT(*) OVER (PARTITION BY mag_bin, color_bin) AS cell_count
    FROM cells
)
SELECT *
FROM ranked
WHERE rank_in_cell = 1 AND cell_count >= {min_count}
ORDER BY mag_bin, color_bin
""".format(distance_modulus=distance_modulus, **gallery_limits)

The `JOIN` to `Photoz` is an inner join here: a galaxy without k-corrections has no rest-frame color or absolute magnitude, so it has no place in the diagram.

In [ ]:
%%time
gallery = CasJobs.executeQuery(
    sql=gallery_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
print(f"{len(gallery)} cells, {gallery['cell_count'].sum():,} galaxies")

In [ ]:
%%time
gallery_images = [
    cutout(galaxy, gallery_limits["width"]) for galaxy in gallery.itertuples()
]

### The gallery

Each galaxy sits on a tile whose shade shows how many galaxies share its cell. Luminosity grows to the left, which is the way astronomers usually draw the diagram.

In [ ]:
mag_step = gallery_limits["mag_step"]
color_step = gallery_limits["color_step"]
mag_low = gallery_limits["mag_min"] + gallery["mag_bin"] * mag_step
color_low = gallery_limits["color_min"] + gallery["color_bin"] * color_step
count_cmap = LinearSegmentedColormap.from_list("tile", ["0.1", "0.6"])
count_norm = LogNorm(
    vmin=gallery_limits["min_count"], vmax=gallery["cell_count"].max()
)
margin = 0.06
mag_range = gallery_limits["mag_max"] - gallery_limits["mag_min"]
color_range = gallery_limits["color_max"] - gallery_limits["color_min"]
# Height of the grid of cells over its width.
grid_ratio = (color_range / color_step) / (mag_range / mag_step)

with plt.style.context(settings.DARK_STYLE):
    fig, ax = plt.subplots(
        figsize=(settings.FIGURE_WIDTH, grid_ratio * settings.FIGURE_WIDTH)
    )
    cells = zip(gallery_images, gallery["cell_count"], mag_low, color_low)
    for image, count, x, y in cells:
        tile_color = count_cmap(count_norm(count))
        tile = Rectangle(
            (x, y), mag_step, color_step, color=tile_color, zorder=0
        )
        ax.add_patch(tile)
        # The magnitude axis runs backwards, so the image's left edge
        # lies at the fainter (larger) magnitude of the cell.
        extent = (
            x + (1 - margin) * mag_step,
            x + margin * mag_step,
            y + margin * color_step,
            y + (1 - margin) * color_step,
        )
        ax.imshow(image, extent=extent, aspect="auto", zorder=1)
    ax.set_xlim(gallery_limits["mag_max"], gallery_limits["mag_min"])
    ax.set_ylim(gallery_limits["color_min"], gallery_limits["color_max"])
    ax.set_aspect(mag_step / color_step)
    ax.set_xlabel("absolute magnitude $M_r$")
    ax.set_ylabel("rest-frame color $g - r$")
    # An inset of the axes keeps the colorbar as tall as the axes, which
    # their fixed aspect ratio makes shorter than the layout's space.
    colorbar_ax = ax.inset_axes([1.04, 0, 0.03, 1])
    fig.colorbar(
        ScalarMappable(norm=count_norm, cmap=count_cmap),
        cax=colorbar_ax,
        label="galaxies in the cell",
    )
    plt.show()

What to look for:

- **Red sequence.** Toward the upper right, at $g - r \approx 0.7$–$0.9$, sit smooth, yellowish galaxies without spiral arms: ellipticals and lenticulars, whose stars are old. The cells brighter than $M_r = -22.5$ hold only red galaxies.
- **Blue cloud.** Below $g - r \approx 0.6$ the galaxies are spirals with blue arms and knots of young stars; toward the faint end they become small, irregular galaxies.
- **The overlap.** Around $g - r \approx 0.6$–$0.7$, spirals with large yellow bulges mix with smooth galaxies.
- **Dust.** Many of the red cells hold thin disks seen edge-on rather than ellipticals. Dust in a disk dims and reddens the light that passes through it, and an edge-on disk puts the most dust in front of its stars, so a spiral can look as red as an elliptical. Color alone does not reveal shape.
- **Counts follow the survey.** The Main Galaxy Sample holds galaxies brighter than $r = 17.77$ in apparent magnitude. A luminous galaxy passes this limit anywhere in the redshift range, a faint one only nearby, so the tiles undercount faint galaxies; in a fixed volume of space, faint galaxies outnumber luminous ones.
- **Measurement errors.** Catalog values are measurements, and some go wrong. The smooth, yellow galaxy in the bluest row has a measured rest-frame $g - r$ of $0.16$, which does not describe the galaxy in the picture. The faint smudge near $M_r = -18.75$, $g - r = 0.75$ lies in the glow of a much brighter neighbor, in the lower left of its frame, whose light probably inflated the smudge's measured size and with it the frame. Wrong values scatter galaxies out of the crowded cells, so a sparse cell at the edge of the diagram is the likeliest place to find one.